In [7]:
from utils import vector_db
from langchain_cohere import CohereRerank

vectore_store = vector_db.connect_db()
query = "What is bom?"
vector_retriever = vectore_store.as_retriever(search_kwargs={"k": 15})
documents = vector_retriever.invoke(query)
documents

Loading existing vector store from C:\Users\wengshang.hoo\Desktop\pp\learning-note02\rag\1. Basic Rag With Chroma\db...


[Document(id='02a5669b-03f4-4ea6-875b-bdcc29320974', metadata={'source': 'C:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\bom.txt'}, page_content='A bill of materials or product structure (sometimes bill of material, BOM or associated list) is a list of the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and the quantities of each needed to manufacture an end product. A BOM may be used for communication between manufacturing partners or confined to a single manufacturing plant. A bill of materials is often tied to a production order whose issuance may generate reservations for components in the bill of materials that are in stock and requisitions for components that are not in stock.\n\nThe first hierarchical databases were developed for automating bills of materials for manufacturing organizations in the early 1960s. At present, this BOM is used as a database to identify the many parts and their codes in autom

In [ ]:
cohere_api_key = ""

reranker = CohereRerank(model="rerank-english-v3.0", top_n=5, cohere_api_key=cohere_api_key)

# Rerank the retrieved documents
reranked_docs = reranker.compress_documents(documents, query)

for i, doc in enumerate(reranked_docs, 1):
    print(f"{i:2d}. {doc.page_content}")

 1. Multi-level BOM
A multi-level bill of materials (BOM), referred to as an indented BOM, is a bill of materials that lists the assemblies, components, and parts required to make a product in a parent-child, top-down method. It provides a display of all items that are in parent-children relationships. When an item is a sub-component, of a (parent) component, it can in-turn have its own child components, and so on. The resulting top-level BOM (item number) would include children; a mix of finished sub-assemblies, various parts and raw materials. A multi-level structure can be illustrated by a tree with several levels. In contrast, a single-level structure only consists of one level of children in components, assemblies and material.
 2. A bill of materials or product structure (sometimes bill of material, BOM or associated list) is a list of the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and the quantities of each needed to manufacture an end product

In [9]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2:1b")

combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in reranked_docs])}

Please provide a clear, helpful answer using only the information from these documents."""

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

result = llm.invoke(messages)
print("Generated Response:")
print(result.content)

Generated Response:
Based on the provided documents, a Bom refers to a bill of materials or product structure that lists the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and quantities needed to manufacture an end product.

In simple terms, Bom is short for Bill of Materials, which is a list of all the components required to make a product. It can be used in various industries such as manufacturing, electronics, and process industries, and is often used for communication between manufacturing partners or confined to a single manufacturing plant.
